# Predict ages with the trained Cannon models

Apply a saved one-step Cannon model (from `train_rgb.ipynb` /
`train_rc.ipynb`) to a sample of stars that is **already** in the matching
evolutionary state -- no RC/RGB classification happens here. Set `STATE`
and `SAMPLE_PATH` below; everything else follows.

The sample defaults to the APOKASC training parquet itself, which is useful
as an end-to-end check of the normalization + test step -- but remember the
final model was trained on those very stars, so agreement there is
optimistic by construction. The honest per-star numbers for APOKASC are the
k-fold out-of-fold predictions inside the training notebooks; this notebook
is for samples the model has not seen (e.g. the bulge targets).

In [1]:
# --- JAX backend selection with CPU fallback (same probe as train_rgb) ---
# The continuum normalization and the Cannon test step run on JAX; a tiny
# matmul checks the GPU backend can actually COMPILE, falling back to CPU
# otherwise.
import jax
import jax.numpy as jnp

try:
    (jnp.ones((8, 8)) @ jnp.ones((8, 8))).block_until_ready()
    print("JAX backend:", jax.default_backend())
except Exception as exc:
    print(f"GPU compile failed ({type(exc).__name__}) -- falling back to CPU")
    jax.config.update("jax_default_device", jax.devices("cpu")[0])
    (jnp.ones((8, 8)) @ jnp.ones((8, 8))).block_until_ready()
    print("JAX backend: CPU (fallback)")

JAX backend: gpu


In [6]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import thecannon as tc
from thecannon import continuum

# scripts/ is not an installed package -- put the repo root on the path.
repo = Path('/home/100/mj8805/scr_mk27/AnniesLasso')
sys.path.insert(0, str(repo))

# apply_model builds the age catalogue (cannon labels + errors with the
# scaled-covariance correction, linear age with log-normal errors, chi^2).
from scripts.apply_rgb_ages import RGB, HEB, apply_model

In [ ]:
STATE = "rc"                       # "rgb" or "rc"

SRC = Path("/home/100/mj8805/scr_mk27/")
DATA = SRC / "bulge-ages-and-orbits/data"

MODEL_PATH = SRC / f"trained_cannon_model_{STATE}.pkl"
# The sample MUST already be pure {STATE}: the model is only valid on the
# population it was trained on. Swap for e.g. the bulge sample.
SAMPLE_PATH = DATA / f"bulge_{STATE}_spectra.parquet"
OUT_PATH = DATA / f"cannon_ages_{STATE}.parquet"

TARGET = {"rgb": RGB, "rc": HEB}[STATE]
REF_AGE_COL = {"rgb": "logAgeRGB", "rc": "logAgeRC"}[STATE]
COLOR = {"rgb": "coral", "rc": "steelblue"}[STATE]

In [13]:
model = tc.CannonModel.read(str(MODEL_PATH))

In [14]:
model

In [15]:
model = tc.CannonModel.read(str(MODEL_PATH))
assert model.is_trained
print("labels:", ", ".join(model.vectorizer.label_names))

labels: raw_teff, raw_logg, raw_fe_h, mg_fe, c_n, al_fe, ce_fe, logAgeRGB


In [16]:
sample = pd.read_parquet(SAMPLE_PATH)
print(f"{len(sample)} stars from {SAMPLE_PATH.name}")

# The wavelength/flux/ivar columns hold one array PER ROW; stack them into a
# 1-D dispersion and 2-D (n_stars, n_pixels) arrays, dropping rows whose
# arrays have the wrong size. Everything row-aligned (the catalogue included)
# is built only AFTER the drop + reset_index (see train_rgb.ipynb for what
# happens otherwise).
def to_array(x):
    if isinstance(x, str):
        return np.array(x.strip("[] \n").split(","), dtype=float)
    arr = np.asarray(x, dtype=float)
    return arr[None] if arr.ndim == 0 else arr

dispersion = to_array(sample["wavelength"].iloc[0])

flux_arrays = [to_array(f) for f in sample["flux"]]
ivar_arrays = [to_array(i) for i in sample["ivar"]]
bad_rows = {i for i, (f, v) in enumerate(zip(flux_arrays, ivar_arrays))
            if f.size != dispersion.size or v.size != dispersion.size}
if bad_rows:
    print(f"dropping {len(bad_rows)} spectra with the wrong pixel count")
    sample = sample.drop(sample.index[list(bad_rows)]).reset_index(drop=True)
    flux_arrays = [a for i, a in enumerate(flux_arrays) if i not in bad_rows]
    ivar_arrays = [a for i, a in enumerate(ivar_arrays) if i not in bad_rows]

flux = np.vstack(flux_arrays)
ivar = np.vstack(ivar_arrays)
assert flux.shape == ivar.shape == (len(sample), dispersion.size)

if model.dispersion is not None:
    assert np.asarray(model.dispersion).size == dispersion.size, \
        "sample and model are on different wavelength grids"
print(flux.shape)

: 

: 

: 

In [ ]:
# Pseudo-continuum-normalize identically to training -- the model is only
# valid on spectra normalized with the same pixel list and chip regions.
continuum_pixels = np.loadtxt(DATA / "continuum.list", dtype=int, comments="#")
continuum_pixels = continuum_pixels[continuum_pixels < dispersion.size]
APOGEE_REGIONS = ([15090, 15822], [15823, 16451], [16452, 16971])

norm_flux, norm_ivar, _, _ = continuum.normalize(
    jnp.array(dispersion), jnp.array(flux), jnp.array(ivar),
    jnp.array(continuum_pixels),
    L=1400, order=3,
    regions=[jnp.array(r) for r in APOGEE_REGIONS],
)
print("median normalized flux (should ~1):",
      float(np.median(np.asarray(norm_flux)[ivar > 0])))

In [ ]:
# Test step + catalogue. The state is known a priori for every star, so the
# provenance columns are constant ("seismic", no classifier probability).
catalogue = apply_model(
    model, sample, norm_flux, norm_ivar,
    source=np.full(len(sample), "seismic"),
    state_proba=np.full(len(sample), np.nan),
    target=TARGET, test_batch_size=32)
assert len(catalogue) == len(sample)
catalogue.head()

In [ ]:
physical = ~catalogue["flag_unphysical_age"]
age = catalogue.loc[physical, "age_gyr"]
print(f"median age {age.median():.2f} Gyr | "
      f"16th-84th {age.quantile(0.16):.2f}-{age.quantile(0.84):.2f} Gyr | "
      f"{int((~physical).sum())} unphysical (outside 0-20 Gyr)")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].hist(age, bins=np.linspace(0, 14, 29), color=COLOR)
axes[0].set_xlabel("Cannon age [Gyr]")
axes[0].set_ylabel("stars")
axes[1].hist(catalogue["r_chi_sq"], bins=np.linspace(0, 10, 41), color=COLOR)
axes[1].set_xlabel(r"reduced $\chi^2$ of the spectral fit")
axes[1].set_title(f"median {catalogue['r_chi_sq'].median():.2f}")
plt.tight_layout()

## Against the reference ages (when the sample has them)

Skipped automatically for samples without a seismic age column. On the
APOKASC default this is an **in-sample** comparison (the final model was
trained on these stars) -- read it as a sanity check on the pipeline, not
as the performance estimate; that is the k-fold section of the training
notebooks. Same conventions as there: color = test-step reduced chi^2,
robust bias/scatter (median and 1.48*MAD) over all finite points.

In [ ]:
if REF_AGE_COL in sample.columns:
    from matplotlib.colors import LogNorm

    ref = sample[REF_AGE_COL].to_numpy(dtype=float)
    pred = catalogue[f"{REF_AGE_COL}_cannon"].to_numpy()
    chi2 = catalogue["r_chi_sq"].to_numpy()
    ok = np.isfinite(ref) & np.isfinite(pred)
    chi_lo, chi_hi = np.percentile(chi2[ok & (chi2 > 0)], [2, 98])
    chi_norm = LogNorm(vmin=chi_lo, vmax=chi_hi)

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2),
                             constrained_layout=True)
    for ax, linear in zip(axes, (False, True)):
        x, y = (10**ref[ok], 10**pred[ok]) if linear else (ref[ok], pred[ok])
        d = y - x
        bias = np.median(d)
        scatter = 1.4826 * np.median(np.abs(d - bias))
        sc = ax.scatter(x, y, c=chi2[ok], s=10, cmap="viridis",
                        norm=chi_norm)
        lims = [min(x.min(), y.min()), max(x.max(), y.max())]
        ax.plot(lims, lims, "-", color="r", lw=1)
        ax.set_xlim(lims); ax.set_ylim(lims)
        name = "Age [Gyr]" if linear else f"{REF_AGE_COL} [dex]"
        ax.set_xlabel(f"{name} (reference)")
        ax.set_ylabel(f"{name} (Cannon)")
        ax.set_title(f"bias={bias:+.3f}, scatter={scatter:.3f}")
    fig.colorbar(sc, ax=axes, shrink=0.8,
                 label=r"reduced $\chi^2$ of the spectral fit")
else:
    print(f"no {REF_AGE_COL} column in the sample -- nothing to compare to")

In [ ]:
catalogue.to_parquet(OUT_PATH, index=False)
print(f"wrote {len(catalogue)}-star age catalogue to {OUT_PATH}")